In [21]:
import os
import cv2
import numpy as np
import random
from tqdm import tqdm
from pathlib import Path

random.seed(69)  # for reproducibility

BASE_DIR = Path("D:/Projects/Masters/Data")
DATASET_ROOT     = BASE_DIR / "AR_dataset"
BEE_DATASET_ROOT = BASE_DIR / "DET_data_sliced_split"
OUTPUT_DIR       = BASE_DIR / "AR_merged_dataset"
CONTEXT_SIZE     = 224                 # adjust this if needed

neutral_RATIO = 0.25  # neutral will be 25% of total dataset

In [22]:
ACTION_CLASSES = sorted([
    f for f in os.listdir(DATASET_ROOT)
    if os.path.isdir(os.path.join(DATASET_ROOT, f))
])

print("Detected action classes:")
for idx, action in enumerate(ACTION_CLASSES):
    print(f"  {idx} → {action}")

Detected action classes:
  0 → fanning
  1 → trophallaxis


In [23]:
# ──── Helper Functions 1 ────
def pad_to_square(crop, target_size):
    """Pad a crop to target_size x target_size using reflect padding.
    FIX: ensures every crop is exactly 320x320 regardless of border proximity.
    """
    h, w = crop.shape[:2]
    pad_bottom = target_size - h
    pad_right  = target_size - w
    # Reflect is safer than black padding — avoids introducing artificial edges
    crop = cv2.copyMakeBorder(crop, 0, pad_bottom, 0, pad_right,
                              cv2.BORDER_REFLECT)
    return crop
def get_crop(frame, cx_norm, cy_norm, context_size=CONTEXT_SIZE):
    """Extract a fixed-size crop centred on a normalised bbox centre.
    Returns the padded crop, or None if something went wrong.
    """
    h, w = frame.shape[:2]
    half = context_size // 2

    center_x = int(cx_norm * w)
    center_y = int(cy_norm * h)

    x1 = max(0, center_x - half)
    y1 = max(0, center_y - half)
    x2 = min(w, center_x + half)
    y2 = min(h, center_y + half)

    crop = frame[y1:y2, x1:x2]
    if crop.size == 0:
        return None

    # pad if the bee was near a border so the crop is always context_size x context_size
    if crop.shape[0] < context_size or crop.shape[1] < context_size:
        crop = pad_to_square(crop, context_size)

    return crop
def audit_split(action_folder, split):
    images_path = os.path.join(DATASET_ROOT, action_folder, split, "images")
    labels_path = os.path.join(DATASET_ROOT, action_folder, split, "labels")

    annotated   = []
    unannotated = []

    for img_file in os.listdir(images_path):
        stem       = os.path.splitext(img_file)[0]
        label_file = os.path.join(labels_path, stem + ".txt")

        is_annotated = False
        if os.path.exists(label_file):
            with open(label_file, "r") as f:
                line = f.readline().strip().split()
            if len(line) >= 5:
                is_annotated = True

        if is_annotated:
            annotated.append(img_file)
        else:
            unannotated.append(img_file)

    return annotated, unannotated
def audit_all_splits():
    summary = {}
    for action in tqdm(ACTION_CLASSES, desc="Auditing dataset"):
        summary[action] = {}
        for split in ["train", "val"]:
            annotated, unannotated = audit_split(action, split)
            summary[action][split] = {
                "annotated":   annotated,
                "unannotated": unannotated
            }
    return summary

In [24]:
# ──── Helper Functions 2 ────
def crop_and_save(img_path, label_path, output_path, context_size=CONTEXT_SIZE):
    frame = cv2.imread(img_path)
    if frame is None:
        return False
    
    h, w = frame.shape[:2]
    
    with open(label_path, "r") as f:
        line = f.readline().strip().split()
    
    if len(line) < 5:
        return False
    
    _, cx, cy, bw, bh = map(float, line[:5])
    
    center_x = int(cx * w)
    center_y = int(cy * h)
    half = context_size // 2
    
    x1 = max(0, center_x - half)
    y1 = max(0, center_y - half)
    x2 = min(w, center_x + half)
    y2 = min(h, center_y + half)
    
    crop = frame[y1:y2, x1:x2]
    
    if crop.size == 0:
        return False
    
    cv2.imwrite(output_path, crop)
    return True
def convert_dataset(summary, context_size=CONTEXT_SIZE):
    """Convert action dataset: crops one patch per annotation line.
    FIX: now reads ALL annotation lines per image, not just the first.
    """
    stats  = {action: {"train": 0, "val": 0} for action in ACTION_CLASSES}
    failed = []

    total = sum(
        len(summary[action][split]["annotated"])
        for action in ACTION_CLASSES
        for split in ["train", "val"]
    )

    with tqdm(total=total, desc="Converting action crops", unit="img") as pbar:
        for action in ACTION_CLASSES:
            for split in ["train", "val"]:
                for img_file in summary[action][split]["annotated"]:
                    stem        = os.path.splitext(img_file)[0]
                    img_path    = os.path.join(DATASET_ROOT, action, split, "images", img_file)
                    label_path  = os.path.join(DATASET_ROOT, action, split, "labels", stem + ".txt")

                    frame = cv2.imread(img_path)
                    if frame is None:
                        failed.append(img_path)
                        pbar.update(1)
                        continue

                    with open(label_path, "r") as f:
                        lines = f.readlines()

                    # FIX: iterate over every annotation in the label file
                    for ann_idx, line in enumerate(lines):
                        parts = line.strip().split()
                        if len(parts) < 5:
                            continue

                        _, cx, cy, bw, bh = map(float, parts[:5])
                        crop = get_crop(frame, cx, cy, context_size)

                        if crop is None:
                            failed.append(img_path)
                            continue

                        # Include ann_idx in filename so multiple bees per frame don't collide
                        out_name    = f"{action}_{stem}_{ann_idx}.jpg"
                        output_path = os.path.join(OUTPUT_DIR, split, action, out_name)
                        cv2.imwrite(output_path, crop)
                        stats[action][split] += 1

                    pbar.update(1)
                    pbar.set_postfix({action: stats[action][split]})

    return stats, failed
def generate_neutral_samples(split, limit, context_size=CONTEXT_SIZE):
    """Sample bee crops from the detection dataset as the 'neutral' class.
    FIX: reports clearly when the source dataset is exhausted before the limit is reached.
    """
    images_path = os.path.join(BEE_DATASET_ROOT, split, "images")
    labels_path = os.path.join(BEE_DATASET_ROOT, split, "labels")
    output_path = os.path.join(OUTPUT_DIR, split, "neutral")

    all_images = [f for f in os.listdir(images_path) if f.endswith((".jpg", ".png"))]
    random.shuffle(all_images)  # seed set globally in Cell 1

    saved  = 0
    failed = 0

    for img_file in tqdm(all_images, desc=f"neutral {split}"):
        if saved >= limit:
            break

        stem       = os.path.splitext(img_file)[0]
        label_file = os.path.join(labels_path, stem + ".txt")

        if not os.path.exists(label_file):
            continue

        frame = cv2.imread(os.path.join(images_path, img_file))
        if frame is None:
            continue

        with open(label_file, "r") as f:
            lines = f.readlines()

        for ann_idx, line in enumerate(lines):
            if saved >= limit:
                break

            parts = line.strip().split()
            if len(parts) < 5:
                continue

            _, cx, cy, bw, bh = map(float, parts[:5])
            crop = get_crop(frame, cx, cy, context_size)

            if crop is None:
                failed += 1
                continue

            out_name = f"neutral_{stem}_{ann_idx}_{saved}.jpg"
            cv2.imwrite(os.path.join(output_path, out_name), crop)
            saved += 1

    # FIX: clearly warn if the source was exhausted before the limit
    if saved < limit:
        print(f"WARNING: {split} neutral exhausted source images. "
              f"Saved {saved}/{limit} — consider augmenting or accepting the imbalance.")
    else:
        print(f"{split} neutral → saved: {saved} | failed: {failed}")

    return saved

In [25]:
for split in ["train", "val"]:
    for action in ACTION_CLASSES:
        os.makedirs(os.path.join(OUTPUT_DIR, split, action), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, split, "neutral"), exist_ok=True)

print("Created folder structure:")
for split in ["train", "val"]:
    print(f"\n  {split}/")
    for folder in sorted(os.listdir(os.path.join(OUTPUT_DIR, split))):
        print(f"    {folder}/")

Created folder structure:

  train/
    fanning/
    neutral/
    trophallaxis/

  val/
    fanning/
    neutral/
    trophallaxis/


In [26]:
summary = audit_all_splits()

total_annotated_train = 0
total_annotated_val   = 0

for action, splits in summary.items():
    print(f"\n{action}")
    for split, data in splits.items():
        count = len(data["annotated"])
        print(f"  {split} → {count} annotated, {len(data['unannotated'])} unannotated")
        if split == "train":
            total_annotated_train += count
        else:
            total_annotated_val += count

print(f"\nTotal annotated train images : {total_annotated_train}")
print(f"Total annotated val images   : {total_annotated_val}")
print(f"\nNeutral limits will be computed after conversion (based on actual crop counts).")

Auditing dataset: 100%|██████████| 2/2 [02:24<00:00, 72.22s/it] 


fanning
  train → 12199 annotated, 2201 unannotated
  val → 2904 annotated, 696 unannotated

trophallaxis
  train → 13506 annotated, 2372 unannotated
  val → 3324 annotated, 1278 unannotated

Total annotated train images : 25705
Total annotated val images   : 6228

Neutral limits will be computed after conversion (based on actual crop counts).


In [27]:
stats, failed = convert_dataset(summary)

print("\nConversion complete!")
for action, splits in stats.items():
    print(f"  {action}")
    for split, count in splits.items():
        print(f"    {split} → {count} crops")
print(f"\nFailed / skipped: {len(failed)}")

# FIX: compute neutral limits based on actual crops saved, not annotated image count
actual_train = sum(stats[action]["train"] for action in ACTION_CLASSES)
actual_val   = sum(stats[action]["val"]   for action in ACTION_CLASSES)

bg_multiplier = neutral_RATIO / (1 - neutral_RATIO)
neutral_TRAIN_LIMIT = int(actual_train * bg_multiplier)
neutral_VAL_LIMIT   = int(actual_val   * bg_multiplier)

print(f"\nActual action train crops : {actual_train}")
print(f"Actual action val crops   : {actual_val}")
print(f"\nCorrected neutral train limit : {neutral_TRAIN_LIMIT} (25% of total)")
print(f"Corrected neutral val limit   : {neutral_VAL_LIMIT} (25% of total)")

Converting action crops: 100%|██████████| 31933/31933 [15:49<00:00, 33.62img/s, trophallaxis=3401] 


Conversion complete!
  fanning
    train → 47773 crops
    val → 9825 crops
  trophallaxis
    train → 18422 crops
    val → 3401 crops

Failed / skipped: 0

Actual action train crops : 66195
Actual action val crops   : 13226

Corrected neutral train limit : 22065 (25% of total)
Corrected neutral val limit   : 4408 (25% of total)


In [28]:
bg_train = generate_neutral_samples("train", neutral_TRAIN_LIMIT)
bg_val   = generate_neutral_samples("val",   neutral_VAL_LIMIT)

# Final class distribution report
print("\n--- Final dataset summary ---")
for split in ["train", "val"]:
    print(f"\n{split}/")
    total = 0
    for folder in sorted(os.listdir(os.path.join(OUTPUT_DIR, split))):
        n = len(os.listdir(os.path.join(OUTPUT_DIR, split, folder)))
        print(f"  {folder:20s}: {n:>6} images")
        total += n
    print(f"  {'TOTAL':20s}: {total:>6} images")

neutral train:  32%|███▏      | 6774/21000 [02:36<05:29, 43.22it/s]


train neutral → saved: 22065 | failed: 0


neutral val:  31%|███       | 1395/4500 [00:29<01:06, 46.75it/s]


val neutral → saved: 4408 | failed: 0

--- Final dataset summary ---

train/
  fanning             :  47773 images
  neutral             :  22065 images
  trophallaxis        :  18422 images
  TOTAL               :  88260 images

val/
  fanning             :   9825 images
  neutral             :   4408 images
  trophallaxis        :   3401 images
  TOTAL               :  17634 images


In [29]:
# Visually verify crops from each class
for folder in os.listdir(os.path.join(OUTPUT_DIR, "train")):
    folder_path = os.path.join(OUTPUT_DIR, "train", folder)
    samples = os.listdir(folder_path)[:2]
    for s in samples:
        img = cv2.imread(os.path.join(folder_path, s))
        if img is not None:
            cv2.imshow(f"{folder} - {s}", img)

cv2.waitKey(0)
cv2.destroyAllWindows()